# Deep Learning and Medical Imaging — Fine-tuning a Pre-trained Model (Radiology)

In this notebook we will train a model to classify small images patches of MR data into tumor or non-tumor:

1. We start from a model that has **already been trained** on a large natural-image dataset (ImageNet) and only adapt it to our medical task. This is called **fine-tuning**.
2. We pay closer attention to a handful of decisions that often matter more than the architecture itself: **how data is split**, **how patches relate to the underlying image**, and **what to do when the model under- or over-fits**.

Throughout the notebook you'll see &#9881; **Try it** boxes (suggestions to experiment) and &#128172; **Discussion** boxes (questions to talk about with your peers and the teachers).

## Problem reminder: classify a patch

We split the MR image into small fixed-size **patches** (32×32 pixels). For each patch we want to predict one of two classes:

* `0` = non-tumor
* `1` = tumor

The model produces two real numbers (one per class). Applying the **softmax** function turns these numbers into a valid probability distribution over the two classes:

$$p(y=i \mid x) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

We pick the class with the highest probability as our prediction. This is what is meant by **structured output**: the network's output is constrained to look like a probability distribution over a fixed, known set of categories. Compare this to a language model that produces free-form text — much harder to evaluate, much harder to act on automatically.

Let's have a look at the data. Below you see an overview of an MR brain image, one with the outlines and tumor manually annotated. 

<img src="https://raw.githubusercontent.com/eseaflower/cmiv-ai-course/master/notebooks/figures/image_MR_brain.png" alt="example MR image" style="height: 300px;"/>
<img src="https://raw.githubusercontent.com/eseaflower/cmiv-ai-course/master/notebooks/figures/image_MR_brain_outline_all.png" alt="example MR image with outlines" style="height: 300px;"/>

Patches are created with a grid across the area inside the marked regions, here 32x32px. The patch is labelled by the **center pixel** of the patch: if it falls inside the lesion, the whole patch is positive; otherwise negative. By running the classifier over a sliding grid of patches we get a coarse segmentation map.

<img src="https://raw.githubusercontent.com/eseaflower/cmiv-ai-course/master/notebooks/figures/image_MR_brain_outline_all_grid.png" alt="patch grid" style="height: 300px;"/>
<img src="https://raw.githubusercontent.com/eseaflower/cmiv-ai-course/master/notebooks/figures/image_MR_brain_outline_all_labels.png" alt="patch labels" style="height: 300px;"/>


&#128172; **Discussion — patches as a representation**

* The patch only sees a 32×32 window. As a radiologist you usually look at the whole slice — and often several slices. What information is the model losing?
* What would change if we used 16×16 or 64×64 patches instead?
* Could you think of a case where **two patches that look identical** should get different labels?

## Setup

In [ ]:
# In Colab, torch and torchvision are pre-installed. We just make sure scikit-learn is available.
%pip install -q scikit-learn matplotlib

In [ ]:
import os, glob, math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}")
print(f"Using device: {device}")

In [ ]:
# Download the MR data (run once; skip if you already have a 'data' folder)
!wget -q -O data.tar https://github.com/eseaflower/cmiv-ai-course/raw/master/data.tar
!tar -xf data.tar
print('Done')

## 1. Loading the data and splitting it the right way

Before we touch any model we make sure our data is split into three **sets**:

| Set | Used for | Seen by the model? |
| --- | --- | --- |
| **Train** | Updating the weights | Yes — every batch |
| **Validation** | Choosing hyperparameters / spotting overfitting | Indirectly — we make descisions from it |
| **Test** | Reporting the final accuracy | **Only once, at the end** |

If you ever look at the test set during model development, it stops being a test set and starts being a second validation set.

In [ ]:
def load_all_patches():
    rootdir = os.path.abspath(os.getcwd())
    datadir = os.path.join(rootdir, 'data', 'MR')
    pos_filenames = sorted(glob.glob(os.path.join(datadir, 'positive_images', '*.jpg')))
    neg_filenames = sorted(glob.glob(os.path.join(datadir, 'negative_images', '*.jpg')))

    pos_images = [matplotlib.image.imread(f) for f in pos_filenames]
    neg_images = [matplotlib.image.imread(f) for f in neg_filenames]

    X = np.vstack([np.array(pos_images, dtype=np.float32),
                   np.array(neg_images, dtype=np.float32)])
    y = np.array([1] * len(pos_images) + [0] * len(neg_images), dtype=np.int64)
    return X, y

X, y = load_all_patches()
print(f'All patches: X={X.shape}, y={y.shape}')
print(f'  positive (tumor):     {(y==1).sum()}')
print(f'  negative (non-tumor): {(y==0).sum()}')

In [ ]:
SEED = 42

# First split off a held-out test set (20% of all patches)
X_trv, X_test, y_trv, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)

# Then split the remaining 80% into train (64%) and validation (16%)
X_train, X_val, y_train, y_val = train_test_split(
    X_trv, y_trv, test_size=0.20, stratify=y_trv, random_state=SEED)

print(f'Train:      X={X_train.shape}, positives={(y_train==1).sum():>5} / {len(y_train)}')
print(f'Validation: X={X_val.shape}, positives={(y_val==1).sum():>5} / {len(y_val)}')
print(f'Test:       X={X_test.shape}, positives={(y_test==1).sum():>5} / {len(y_test)}')

&#128172; **Discussion — about the split we just did**

* `stratify=y` makes sure each set has the same tumor/non-tumor ratio. Why does that matter for **classification accuracy** as a reported number?
* We split **patches**, not **patients**. The dataset doesn't expose patient or slice IDs, so we have no way to guarantee that patches from the same patient — or even the same slice — don't end up in both train and test. What does this do to the test-set accuracy we will report?
* If you had patient IDs, splitting on them would stop the same patient's patches from showing up in both train and test. But your effective test-set size would drop from thousands of patches to a few dozen patients — what does that do to how much you can trust the reported accuracy?
* Patient is one source of correlated samples. What other source might still leak across train and test even after a patient-level split? (Hint: think about where the images came from.)

&#9881; **Try it later** — once you have a working model, come back, change `SEED`, retrain, and see how much your test accuracy moves. That movement is your noise floor.

In [ ]:
def plot_patches(X, y, y_true=None, to_plot=None):
    to_plot = to_plot or len(X)
    plt.figure(figsize=(16, 2))
    for i in range(to_plot):
        plt.subplot(1, to_plot, i + 1)
        plt.imshow(X[i].reshape((32, 32)), interpolation='nearest', cmap='gray')
        plt.text(0, 0, y[i], color='black',
                 bbox=dict(facecolor='white', alpha=1))
        if y_true is not None:
            plt.text(0, 32, y_true[i], color='black',
                     bbox=dict(facecolor='white', alpha=1))
        plt.axis('off')
    plt.show()

print('Some training patches (label in upper-left; 1 = tumor, 0 = non-tumor):')
plot_patches(X_train, y_train, to_plot=10)

## 2. Wrapping patches as PyTorch tensors

Pre-trained ImageNet models expect **3-channel RGB images** of a particular size, normalized in a particular way. Our MR patches are **1-channel grayscale 32×32**. We bridge the gap inside a `Dataset`:

1. Cast the patch to a tensor in `[0, 1]`.
2. Replicate the single grayscale channel three times → fake "RGB".
3. Resize from 32×32 to 96×96 (the smallest size ImageNet backbones accept comfortably; bigger means slower).
4. Apply ImageNet's mean/std normalization so the input distribution matches what the backbone expects.

This bridging step is small but easy to get wrong. If you ever fine-tune a pre-trained model and accuracy refuses to go above chance, **check the normalization first**.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
INPUT_SIZE    = 96

class MRPatchDataset(Dataset):
    """Wrap (X, y) numpy arrays as a PyTorch dataset that yields ImageNet-ready tensors."""
    def __init__(self, X, y, augment=None):
        self.X = X
        self.y = y
        self.augment   = augment
        self.resize    = transforms.Resize((INPUT_SIZE, INPUT_SIZE), antialias=True)
        self.normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.X[idx]).unsqueeze(0) / 255.0   # (1, 32, 32) in [0, 1]
        img = img.repeat(3, 1, 1)                                  # (3, 32, 32) replicate gray -> RGB
        img = self.resize(img)                                     # (3, 96, 96)
        if self.augment is not None:
            img = self.augment(img)
        img = self.normalize(img)
        return img, int(self.y[idx])

BATCH_SIZE = 64

train_ds = MRPatchDataset(X_train, y_train)
val_ds   = MRPatchDataset(X_val,   y_val)
test_ds  = MRPatchDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## 3. Building the model

We start from a model that already "knows how to look at images" — a CNN trained on millions of natural images (ImageNet). The deep features it has learned (edges, textures, shapes) are surprisingly transferable, even to domains it has never seen, like radiology.

We then

* **replace** the final classification layer (1000 ImageNet classes → 2 medical classes), and
* **freeze** the rest of the network so we don't accidentally destroy the pre-trained features in the first few noisy gradient steps.

We offer two backbones below. **MobileNetV2** is small and fast (≈3.5M parameters) and is fine on a CPU. **ResNet50** is larger (≈25M parameters), more accurate on ImageNet, and slower. Switch between them by uncommenting the block you want.

In [ ]:
def build_model():
    # --- Option A: MobileNetV2 (small, fast) ---
    backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)
    in_features = backbone.classifier[-1].in_features
    backbone.classifier[-1] = nn.Linear(in_features, 2)

    # --- Option B: ResNet50 (larger, slower). Comment out Option A and uncomment this block to use. ---
    # backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    # in_features = backbone.fc.in_features
    # backbone.fc = nn.Linear(in_features, 2)

    return backbone


def freeze_backbone(model):
    """Freeze every parameter, then unfreeze just the new classifier head."""
    for p in model.parameters():
        p.requires_grad = False
    if isinstance(model, models.MobileNetV2):
        for p in model.classifier.parameters():
            p.requires_grad = True
    else:  # ResNet
        for p in model.fc.parameters():
            p.requires_grad = True


def unfreeze_top_blocks(model, n_blocks=2):
    """Unfreeze the last n blocks of the backbone for fine-tuning stage 2."""
    if isinstance(model, models.MobileNetV2):
        for block in list(model.features.children())[-n_blocks:]:
            for p in block.parameters():
                p.requires_grad = True
    else:  # ResNet
        layers = [model.layer4, model.layer3, model.layer2, model.layer1]
        for layer in layers[:n_blocks]:
            for p in layer.parameters():
                p.requires_grad = True


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


model = build_model().to(device)

print(f'Total parameters:     {sum(p.numel() for p in model.parameters()):>12,}')
print(f'Trainable parameters: {count_trainable(model):>12,}  (head only — backbone frozen)')

## 4. Training

We train in **two stages**:

* **Stage 1 — head only.** With the backbone frozen, only the new classifier learns. This is fast and it stops the (initially random) head from sending nonsense gradients backward into the pre-trained weights.
* **Stage 2 — top blocks unfrozen, tiny learning rate.** Once the head is reasonable, we let the top of the backbone adapt to the medical domain. The small learning rate is crucial: we want to *nudge* the pre-trained features, not overwrite them.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * xb.size(0)
        correct  += (logits.argmax(1) == yb).sum().item()
        total    += xb.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss_sum += loss.item() * xb.size(0)
        correct  += (logits.argmax(1) == yb).sum().item()
        total    += xb.size(0)
    return loss_sum / total, correct / total


def fit(model, train_loader, val_loader, optimizer, epochs, history=None):
    criterion = nn.CrossEntropyLoss()
    history = history or {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
    for _ in range(epochs):
        tl, ta = train_one_epoch(model, train_loader, optimizer, criterion)
        vl, va = evaluate(model, val_loader, criterion)
        history['train_loss'].append(tl); history['train_acc'].append(ta)
        history['val_loss'].append(vl);   history['val_acc'].append(va)
        print(f"Epoch {len(history['train_acc']) - 1:>2}: L: {tl:.4f}  A: {ta:.4f}  VL: {vl:.4f}  VA: {va:.4f}")
    return history


def plot_history(history):
    plt.figure(figsize=(6, 4))
    plt.plot(history['train_acc'], c='r', label='Train')
    plt.plot(history['val_acc'],   c='g', label='Validation')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()
    plt.show()

Let's start with training the 'head' only. You can try and change **learning rate** (how big changes the model makes after each iteration) and the **epochs** (how many times the model sees the entire training set before training finishes).

In [ ]:
# Hyperparameters to tune:
LEARNING_RATE = 1e-3    
EPOCHS = 5


In [ ]:
# Stage 1: freeze backbone, train head only, regular learning rate

freeze_backbone(model)

optimizer_s1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
print('Stage 1 — training the new head with the backbone frozen:')
history = fit(model, train_loader, val_loader, optimizer_s1, epochs=EPOCHS)

Now let's try and train a bit more of the parameters by unfreezing the final blocks (layers) of the backbone network

In [ ]:
# Hyperparameters to tune:
LEARNING_RATE_2 = 1e-5
EPOCHS_2 = 10

In [ ]:
# Stage 2: unfreeze the top of the backbone, tiny learning rate
unfreeze_top_blocks(model, n_blocks=2)
print(f'Trainable parameters after unfreezing: {count_trainable(model):,}')

optimizer_s2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE_2)
print('Stage 2 — fine-tuning the top blocks of the backbone:')
history = fit(model, train_loader, val_loader, optimizer_s2, epochs=EPOCHS_2, history=history)
plot_history(history)

&#128172; **Discussion — pre-trained vs from scratch**

* What do you think would happen if we instead trained a model from scratch? 
* The pre-trained model has never seen an MR image. Why does it still help?
* Is there a domain where transferring from ImageNet would *not* help?

## 5. Underfitting, overfitting, and the importance of data

Look at the training curve from the previous step. Three patterns to recognize:

* **Underfitting** — both train and validation accuracy stay low. The model isn't expressive enough, or you stopped too early.
* **Overfitting** — train accuracy keeps climbing while validation accuracy plateaus or *drops*. The model is memorizing patches it has seen instead of learning the underlying signal.
* **Healthy fit** — train and validation accuracy rise together and stay close.

There are many knobs we can turn to fight overfitting (dropout, weight decay, smaller model, early stopping). The single most powerful one is almost always: **more data, or more variety in the data we already have**. When real new data isn't available, **data augmentation** — synthetically perturbing the images we already have — is the next-best thing.

&#128172; **Discussion — augmentations that make sense for MR**

Pathology images can be flipped, rotated to any angle, and color-jittered to fake stain variation. MR is different. Before running the next cell, decide what to apply:

* **Horizontal flip** — does a flipped MR brain represent a *real* patient you might see? (Hint: think about left/right asymmetric findings.)
* **Vertical flip** — does it make any anatomical sense?
* **Small rotations** — yes/no?
* **Brightness / contrast jitter** — does this correspond to real-world variation between scanners and protocols?
* **Heavy zoom or shear** — would this still be a realistic image of the same finding?

An augmentation that produces images you would never see in clinical practice doesn't help — it teaches the model to be invariant to nonsense.

In [ ]:
# A conservative augmentation pipeline for MR brain patches.
# Try changing it: comment lines out, add ColorJitter, add VerticalFlip, etc.
mr_augment = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    # transforms.ColorJitter(brightness=0.1, contrast=0.1),
    # transforms.RandomVerticalFlip(p=0.5),
])

# Visualize: same patch, 10 different augmentations
sample_X = np.repeat(X_train[0:1], 10, axis=0)
sample_y = np.zeros(10, dtype=np.int64)
viz_ds = MRPatchDataset(sample_X, sample_y, augment=mr_augment)

fig, axes = plt.subplots(1, 10, figsize=(16, 2))
for i, ax in enumerate(axes):
    img, _ = viz_ds[i]
    img_disp = img.permute(1, 2, 0).numpy()
    img_disp = img_disp * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_disp = np.clip(img_disp, 0, 1)
    ax.imshow(img_disp[:, :, 0], cmap='gray'); ax.axis('off')
plt.suptitle('Same patch, 10 random augmentations'); plt.show()

In [ ]:
# Train a fresh model with augmentation so we can compare fairly
model_aug = build_model().to(device)
freeze_backbone(model_aug)

train_ds_aug     = MRPatchDataset(X_train, y_train, augment=mr_augment)
train_loader_aug = DataLoader(train_ds_aug, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

opt = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_aug.parameters()), lr=LEARNING_RATE)
print('Stage 1 with augmentation:')
history_aug = fit(model_aug, train_loader_aug, val_loader, opt, epochs=EPOCHS)

unfreeze_top_blocks(model_aug, n_blocks=2)
opt = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_aug.parameters()), lr=LEARNING_RATE_2)
print('Stage 2 with augmentation:')
history_aug = fit(model_aug, train_loader_aug, val_loader, opt, epochs=EPOCHS_2, history=history_aug)

plot_history(history_aug)

&#9881; **Try it — provoke each pattern**

1. **Force underfitting** — set Stage 1 to 1 epoch, skip Stage 2 entirely. Where does validation accuracy land?
2. **Force overfitting** — turn off augmentation, unfreeze the *whole* backbone from epoch 0 with `lr=1e-3`, train for 30 epochs. Watch train accuracy go up while validation accuracy stalls or falls.
3. **Find a middle ground** — keep two stages, keep augmentation, tune number of unfrozen blocks and epochs.

&#128172; **Discussion — would more data beat all of these tricks?**

## 6. Final evaluation on the test set

Now — and only now — we touch the test set. Accuracy is one number; for a clinical classifier we want to look more carefully at *which kinds* of mistakes the model makes.

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, true = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        probs.append(F.softmax(model(xb), dim=1).cpu().numpy())
        true.append(yb.numpy())
    return np.concatenate(probs, 0), np.concatenate(true, 0)

# Choose the model to evaluate (the augmented one by default)
eval_model = model_aug
probs_test, y_true = predict(eval_model, test_loader)
y_pred = probs_test.argmax(axis=1)

acc = (y_pred == y_true).mean()
print(f'Test accuracy: {acc:.4f}  (n = {len(y_true)})')

In [ ]:
# Confusion matrix — what kind of mistakes is the model making?
cm = np.zeros((2, 2), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

print('Confusion matrix (rows = true, cols = predicted):')
print(f'                    pred=0   pred=1')
print(f'  true=0  (n={ (y_true==0).sum() :>4})   {cm[0,0]:>5}    {cm[0,1]:>5}')
print(f'  true=1  (n={ (y_true==1).sum() :>4})   {cm[1,0]:>5}    {cm[1,1]:>5}')

tn, fp = cm[0]
fn, tp = cm[1]
sens = tp / max(tp + fn, 1)
spec = tn / max(tn + fp, 1)
print(f'\nSensitivity (recall on tumor):     {sens:.3f}')
print(f'Specificity (recall on non-tumor): {spec:.3f}')

In [ ]:
# Predicted-probability histograms — is the model confident or hesitant?
plt.figure(figsize=(8, 4))
plt.hist(probs_test[y_true == 0, 1], bins=30, alpha=0.6, label='True non-tumor')
plt.hist(probs_test[y_true == 1, 1], bins=30, alpha=0.6, label='True tumor')
plt.axvline(0.5, color='k', linestyle='--', label='Default threshold (0.5)')
plt.xlabel('Predicted P(tumor)'); plt.ylabel('Number of patches')
plt.legend(); plt.show()

In [ ]:
# Look at the mistakes
errors = y_pred != y_true
print(f'Misclassified: {errors.sum()} / {len(y_true)}')

to_show = 15
err_idx = np.where(errors)[0][:to_show]
print('Misclassified patches (prediction in upper-left, true label in bottom-left):')
plot_patches(X_test[err_idx], y_pred[err_idx], y_true=y_true[err_idx])

&#128172; **Discussion — interpreting the test result**

* Accuracy was a single number. The confusion matrix split it into two numbers (sensitivity, specificity). Which is more important for a tumor *screening* tool? For a *confirmation* tool?
* The default decision threshold is `P(tumor) > 0.5`. Looking at the probability histograms, where would you move the threshold if you wanted to miss as few tumors as possible — and what would you give up?
* Look at the misclassified patches. Are the model's mistakes the ones a human would also find hard? Are there any that look *trivially* wrong?
* Remember: this test accuracy may be optimistic because patches from the same patient can appear in train and test. How would you re-design the experiment given more time and access to patient IDs?

## 7. Wrap-up

* We started from a model **already trained on millions of images** instead of from scratch — fine-tuning typically converges faster and ends higher than training from scratch.
* We split data into **three** sets and only used the test set once.
* We thought about which augmentations are physically meaningful for MR (and which would be irrelevant).
* We looked beyond plain accuracy to **sensitivity, specificity, and the predicted-probability distribution**.

&#128172; **Final discussion**

* When does it make sense to train from scratch instead of fine-tuning?
* You have a budget of 100 extra annotated patches. Do you spend them on **train**, **validation**, or **test**? Why?
* So-called **foundation models** for medical imaging (CLIP-style, RAD-DINO, MedSAM, …) are trained on huge medical-image collections. How would the workflow in this notebook change if you started from one of those instead of ImageNet?
* For a real product, a patch-level accuracy is not the user-facing metric. What metric would actually matter to the radiologist looking at the slide?